In [ ]:
# Mount Google Drive

# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
! pip install transformers datasets evaluate seqeval -q

In [ ]:
import json

with open('/path/to/your/file', 'r') as f:
    merged_data = json.load(f)

In [ ]:
print(merged_data[0:1])

In [ ]:
import random
import numpy as np
import torch
import os
from sklearn.metrics import classification_report
from datasets import Dataset
from transformers import (
    BertTokenizerFast, BertForTokenClassification,
    TrainingArguments, Trainer,
    DataCollatorForTokenClassification,
    AutoModelForTokenClassification
)

os.environ["WANDB_DISABLED"] = "true"

In [ ]:
# Set seed function
def set_seed(seed):
    torch.backends.cudnn.deterministic = True
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
set_seed(0)

def split_data(data, train_ratio=0.80, dev_ratio=0.10, test_ratio=0.10):
    assert abs(train_ratio + dev_ratio + test_ratio - 1.0) < 1e-6, "Ratios must sum to 1"

    # Shuffle
    random.shuffle(data)

    # Calculate sizes
    total_size = len(data)
    train_size = int(train_ratio * total_size)
    dev_size = int(dev_ratio * total_size)
    test_size = total_size - train_size - dev_size

    # Split data
    train_data = data[:train_size]
    dev_data = data[train_size:train_size + dev_size]
    test_data = data[train_size + dev_size:]

    return train_data, dev_data, test_data

train_data, dev_data, test_data = split_data(merged_data)

In [ ]:
def chunk_example(example, max_len=512):

    tokens = example["tokens"]
    tags_1 = example["ner_tags_1"]
    tags_2 = example["ner_tags_2"]

    chunks = []
    for start_idx in range(0, len(tokens), max_len):
        end_idx = start_idx + max_len

        chunk_tokens = tokens[start_idx:end_idx]
        chunk_tags_1 = tags_1[start_idx:end_idx]
        chunk_tags_2 = tags_2[start_idx:end_idx]

        chunk_example = {
            "id": f"{example['id']}_chunk_{start_idx}",
            "tokens": chunk_tokens,
            "ner_tags_1": chunk_tags_1,
            "ner_tags_2": chunk_tags_2
        }
        chunks.append(chunk_example)

    return chunks

In [ ]:
def chunk_data(data, max_len=512):
    chunked_dataset = []
    for example in data:
        chunked_dataset.extend(chunk_example(example, max_len=max_len))
    return chunked_dataset

# Apply chunking to train, dev, test sets
chunked_train_data = chunk_data(train_data, max_len=512)
chunked_dev_data   = chunk_data(dev_data, max_len=512)
chunked_test_data  = chunk_data(test_data, max_len=512)

print(f"Train examples (original): {len(train_data)} → chunked: {len(chunked_train_data)}")
print(f"Dev examples (original):   {len(dev_data)} → chunked: {len(chunked_dev_data)}")
print(f"Test examples (original):  {len(test_data)} → chunked: {len(chunked_test_data)}")

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained("bert-large-uncased")

In [ ]:
# Function for tokenization and label alignment
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        max_length=512,
        padding="max_length",
        truncation=True,
        is_split_into_words=True)

    labels = []
    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None
    label_ids = []

    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)
        elif word_idx != previous_word_idx:
            label_ids.append(example["ner_tags_2"][word_idx])
        else:
            label_ids.append(-100)
        previous_word_idx = word_idx

    tokenized_inputs["labels"] = label_ids
    return tokenized_inputs

In [ ]:
# Apply tokenization and alignment on each split
tokenized_train_data = [tokenize_and_align_labels(example) for example in chunked_train_data]
tokenized_dev_data = [tokenize_and_align_labels(example) for example in chunked_dev_data]
tokenized_test_data = [tokenize_and_align_labels(example) for example in chunked_test_data]

In [ ]:
train_dataset = Dataset.from_list(tokenized_train_data)
dev_dataset = Dataset.from_list(tokenized_dev_data)
test_dataset = Dataset.from_list(tokenized_test_data)

In [ ]:
print(train_dataset[0]["labels"])

In [ ]:
def check_labels(dataset):
    all_labels = set()
    for example in dataset:
        all_labels.update(example["labels"])  # Collect unique labels

    print(f"Unique label values in dataset: {all_labels}")

check_labels(train_dataset)

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

Train

In [ ]:
id2label = {
    0: "O",
    1: "Implicature",
    2: "Ambiguity",
    3: "Presupposition",
    4: "Explicit"
}
label2id = {
    "O": 0,
    "Implicature": 1,
    "Ambiguity": 2,
    "Presupposition": 3,
    "Explicit": 4
}

In [ ]:
def evaluate_model(trainer, eval_dataset, id2label):
    from transformers import AutoModelForTokenClassification
    from sklearn.metrics import classification_report

    if trainer.state.best_model_checkpoint:
        print(f"Loading best model from checkpoint: {trainer.state.best_model_checkpoint}")
        trainer.model = AutoModelForTokenClassification.from_pretrained(
            trainer.state.best_model_checkpoint
        ).to(trainer.args.device)

    predictions, labels, _ = trainer.predict(eval_dataset)
    preds = np.argmax(predictions, axis=-1)

    true_labels, pred_labels = [], []
    for i, label in enumerate(labels):
        true_labels.extend([id2label[l] for l in label if l != -100])
        pred_labels.extend([id2label[p] for (p, l) in zip(preds[i], label) if l != -100])

    # Print full report with support
    print("\n" + classification_report(true_labels, pred_labels, digits=4))

    # Extract report
    report = classification_report(true_labels, pred_labels, digits=4, output_dict=True)
    macro_f1 = report["macro avg"]["f1-score"]

    scores = {
        "macro": macro_f1,
        "Implicature": report["Implicature"]["f1-score"] if "Implicature" in report else 0.0,
        "Ambiguity": report["Ambiguity"]["f1-score"] if "Ambiguity" in report else 0.0,
        "Presupposition": report["Presupposition"]["f1-score"] if "Presupposition" in report else 0.0,
        "Explicit": report["Explicit"]["f1-score"] if "Explicit" in report else 0.0,
    }

    return scores

In [ ]:
seeds = [42, 123, 2025]
results = []

for seed in seeds:
    print(f"\nTraining with seed {seed}\n")
    set_seed(seed)

    model = BertForTokenClassification.from_pretrained(
        "bert-large-uncased", num_labels=5, id2label=id2label, label2id=label2id
    )

    training_args = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        learning_rate=2e-5,
        eval_strategy="epoch",
        logging_strategy="steps",
        logging_steps=10,
        per_device_train_batch_size=5,
        per_device_eval_batch_size=5,
        num_train_epochs=8,
        weight_decay=0.01,
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        seed=seed,
        report_to=[],
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )

    trainer.train()
    run_scores = evaluate_model(trainer, test_dataset, id2label)
    results.append(run_scores)

In [ ]:
score_types = ["macro", "Implicature", "Ambiguity", "Presupposition", "Explicit"]
summary = {
    score: {
        "mean": np.mean([r[score] for r in results]),
        "std": np.std([r[score] for r in results])
    } for score in score_types
}

print("\n Final Results Across Runs \n")
for i, seed in enumerate(seeds):
    r = results[i]
    print(
        f"Seed {seed}: "
        f"Macro F1 = {r['macro']:.4f}, "
        f"Implicature = {r['Implicature']:.4f}, "
        f"Ambiguity = {r['Ambiguity']:.4f}, "
        f"Presupposition = {r['Presupposition']:.4f}, "
        f"Explicit = {r['Explicit']:.4f}"
    )

print("\n Averaged Results \n")
for score_type in score_types:
    print(f"{score_type} F1: Mean = {summary[score_type]['mean']:.4f}, Std = {summary[score_type]['std']:.4f}")